In [ ]:
# Run from the repository root so that `utils` and the relative data paths resolve
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import scipy.signal as signal
from glob import glob
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pickle
from itertools import groupby
from tqdm import tqdm

from utils.composer import read_pickle, correct_strain

## Example

In [ ]:
# Input folder
signal_folder = "data/raw/signals"  # folder of raw *.pickle recordings (not public)


signal_list = glob(signal_folder + "/*.pickle")
signal_list.sort()
# print(pd.read_pickle(signal_list[0]).keys())

# filtered_list = [i for i in signal_list if i.split("_")[-2] == "20240827110726"]
# filtered_list = [i for i in signal_list if i.split("_")[-2].startswith("20240825")]
filtered_list = []
for i in signal_list:
    ft = pd.to_datetime(i.split("_")[-2], format='%Y%m%d%H%M%S')
    if ft.day == 25 and ft.hour >=8 and ft.hour <= 17:
        filtered_list.append(i)
print(f"Filtered list has length = {len(filtered_list)}")

# Sensors
window_size = 500 # seconds
step = 500 # seconds
print("Overlapping = ", 1 - step / window_size)

# label
df_label = pd.read_csv("data/label_axle_corr.csv", parse_dates=["start", "end"])
# df_label["start"] = df_label["start"] - pd.Timedelta(seconds=3.2)
# df_label["end"] = df_label["end"] - pd.Timedelta(seconds=3.2)



In [ ]:
# Output folder
output_folder = Path("dataset/day_sample")
output_folder.mkdir(parents=True, exist_ok=True)

# Generate samples
for f in filtered_list:
    print(f"Processing {f.split('_')[-2]}")
    dual_np, start, end = read_pickle(f, norm=False)
    dual_np = dual_np[:,250:-250,:]
    start = start + pd.Timedelta(seconds=2.5)
    end = end - pd.Timedelta(seconds=2.5)

    piece_num = (dual_np.shape[1] - (window_size-step)) // step

    for i in range(piece_num):

        # Sample
        sample = dual_np[:, i*step:i*step+window_size, :]
        sample_start = start + pd.Timedelta(seconds=step/100)*i
        sample_end = sample_start + pd.Timedelta(seconds=window_size/100)

        # Label
        within_vehicles = df_label[(df_label["start"] >= sample_start) & (df_label["start"] <= sample_end) |
                                   (df_label["end"] >= sample_start) & (df_label["end"] <= sample_end)]
        reg_label = [0,0,0,0]  # (CL, CR, TL, TR)
        for v in within_vehicles.itertuples():
            lane = 0 if v.x < 7 else 1
            obj_type = v.type
            duration = v.end - v.start

            if v.start >= sample_start and v.end <= sample_end:
                vc = 1
            elif v.start < sample_start and v.end < sample_end:
                vc = (v.end - sample_start) / duration
            elif v.start > sample_start and v.end > sample_end:
                vc = (sample_end - v.start) / duration
            else:
                print(f"Impossible exception: vehicle from {v.start} to {v.end}. This sample is from {sample_start}.")

            
            if obj_type == 1:    # car
                if lane == 0:  # left lane
                    reg_label[0] += vc
                else:           # right lane
                    reg_label[1] += vc
            elif obj_type == 2:  # truck
                if lane == 0:
                    reg_label[2] += vc
                else:
                    reg_label[3] += vc
            else:
                raise ValueError(f"Unknown vehicle type {obj_type}")
            
        sample_name = sample_start.strftime("%Y%m%d%H%M%S")

        # np.save(singal_folder / f"{sample_name}.npy", sample)
        # with open(label_folder / f"{sample_name}.txt", "w") as f:
        #     f.write(" ".join(map(str, reg_label)))
        with open(output_folder / f"{sample_name}.pkl", "wb") as f:
            sample_dict = {
                "signal": sample,
                "label": reg_label,
                # "start": sample_start.strftime("%Y%m%d%H%M%S"),
                # "end": sample_end.strftime("%Y%m%d%H%M%S")
            }
            pickle.dump(sample_dict, f)

        # print(f"Sample {sample_name} saved. Label is {reg_label}.")

In [ ]:
# # Batch norm signals
# sample_list = glob(str(output_folder)+"/*.pkl")

# # Group by hours
# grouped_files = {key: list(group) for key, group in groupby(sample_list, key=lambda x: str(Path(x).stem)[:10])}

# # Norm by group
# for group, file_list in grouped_files.items():
#     print(f"Normalizing group: {group}")
#     norm_batch_signal(file_list)


## Testset

In [ ]:

for day in [30,31]:
    for hour in range(8,18):
        print(f"Processing day {day}, hour {hour}")
        test_list = []
        for i in signal_list:
            ft = pd.to_datetime(i.split("_")[-2], format='%Y%m%d%H%M%S')
            if ft.day == day and ft.hour == hour:
                test_list.append(i)
        test_list.sort()
        print(f"Test list has length = {len(test_list)}")

        test_folder = Path("dataset/test_folder_daytime")
        test_folder.mkdir(parents=True, exist_ok=True)

        # Generate samples
        _, test_start, _ = read_pickle(test_list[0], norm=False, signal_filter=False)
        _, _, test_end = read_pickle(test_list[-1], norm=False, signal_filter=False)

        print(f"Start: {test_start}, End: {test_end}")

        test_signal_list = []
        for f in test_list:
            # print(f"Processing {f.split('_')[-2]}")
            dual_np, _, _ = read_pickle(f, norm=True, signal_filter=False)
            test_signal_list.append(dual_np)

        # Nromalize test signals
        test_complete = np.concatenate(test_signal_list, axis=1)
        test_complete[0] = np.apply_along_axis(signal.detrend, axis=0, arr=test_complete[0])  # remove linear trend
        test_complete[1] = correct_strain(test_complete[1], alpha=0.1, cutoff=2.5, fs=100.0)

        # Trim test signals
        trim_length = (test_complete.shape[1] % window_size) // 2
        print(f"Trim length: {trim_length}")
        test_complete = test_complete[:, trim_length:-trim_length, :]
        if test_complete.shape[1] % window_size != 0:
            test_complete = test_complete[:, :-(test_complete.shape[1] % window_size), :]
        test_start = test_start + pd.Timedelta(seconds=trim_length/100)
        test_end = test_end - pd.Timedelta(seconds=trim_length/100)

        print(f"Signal shape: {test_complete.shape}")

        piece_num = (test_complete.shape[1] - (window_size-step)) // step
        print(f"Piece num: {piece_num}")

        test_label_list = []
        missing_count = 0
        for i in range(piece_num):

            sample_start = test_start + pd.Timedelta(seconds=step/100)*i
            sample_end = sample_start + pd.Timedelta(seconds=window_size/100)

            # print(f"Sample {i}: {sample_start} to {sample_end}")
            # print(f"shift is {shift}")

            # if (sample_end > pd.to_datetime("2024-08-31 06:37:14") and sample_end < pd.to_datetime("2024-08-31 06:42:14")):
            #     print(f"This is {i}")
            #     missing_count += 1
            #     shift = 300

            # Label
            within_vehicles = df_label[(df_label["start"] >= sample_start) & (df_label["start"] <= sample_end) |
                                        (df_label["end"] >= sample_start) & (df_label["end"] <= sample_end)]
            reg_label = [0,0,0,0]  # (CL, CR, TL, TR)
            for v in within_vehicles.itertuples():
                lane = 0 if v.x < 7 else 1
                obj_type = v.type
                duration = v.end - v.start

                if v.start >= sample_start and v.end <= sample_end:
                    vc = 1
                elif v.start < sample_start and v.end < sample_end:
                    vc = (v.end - sample_start) / duration
                elif v.start > sample_start and v.end > sample_end:
                    vc = (sample_end - v.start) / duration
                else:
                    print(f"Impossible exception: vehicle from {v.start} to {v.end}. This sample is from {sample_start}.")

                
                if obj_type == 1:    # car
                    if lane == 0:  # left lane
                        reg_label[0] += vc
                    else:           # right lane
                        reg_label[1] += vc
                elif obj_type == 2:  # truck
                    if lane == 0:
                        reg_label[2] += vc
                    else:
                        reg_label[3] += vc
                else:
                    raise ValueError(f"Unknown vehicle type {obj_type}")
                
            test_label_list.append(reg_label)

            test_label_array = np.array(test_label_list)

        print(f"Test label array has shape {test_label_array.shape}")

        # Save test signals and labels
        with open(test_folder / f"{test_list[0].split('_')[-2]}.pkl", "wb") as f:
            test_dict = {
                "signal": test_complete,
                "label": test_label_array,
                # "start": sample_start.strftime("%Y%m%d%H%M%S"),
                # "end": sample_end.strftime("%Y%m%d%H%M%S")
            }
            pickle.dump(test_dict, f)

print(f"Missing count: {missing_count}")

In [ ]:
with open("dataset/test_folder/20240831170247.pkl", 'rb') as f:  # Open file in binary read mode
    pk = pickle.load(f)
    test_signal = pk["signal"]  # Deserialize the signal data
    test_label = pk["label"]

with open("dataset/test_folder_axle/20240831170247.pkl", 'rb') as f:  # Open file in binary read mode
    pk = pickle.load(f)
    test_signal = pk["signal"]  # Deserialize the signal data
    test_label_axle = pk["label"]
print(np.sum(test_label==test_label_axle))

## Visual

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils.composer import correct_strain
from utils.dataloader import add_gaussian_noise_standardized

# Load one raw recording
signal_path = signal_list[140]  # signal_list is defined in the first cell
print(signal_path)
dual_np = read_pickle(signal_path, norm=False)[0]
dual_np[0] = np.apply_along_axis(signal.detrend, axis=0, arr=dual_np[0])  # remove linear trend
dual_np[1] = correct_strain(dual_np[1], alpha=0.1, cutoff=0.8, fs=100.0)
# print(np.max(np.abs(dual_np[0])))

In [ ]:
signal_plot = dual_np[1,-750:-250,[3]].transpose()  # 500 samples of one strain gauge

# Plot
plt.figure(figsize=(20, 5))
x = np.arange(signal_plot.shape[0])
plt.plot(x, signal_plot)
plt.xlabel("Time")
plt.ylabel("Strain (µε)")
plt.title("Real-time Dynamic Zero Drift Correction using EWMA and Low-pass Filtering")
plt.show()

signal_plot = add_gaussian_noise_standardized(signal_plot, 0.0001)  # add Gaussian noise

# Plot
plt.figure(figsize=(20, 5))
x = np.arange(signal_plot.shape[0])
plt.plot(x, signal_plot)
plt.xlabel("Time")
plt.ylabel("Strain (µε)")
plt.title("Real-time Dynamic Zero Drift Correction using EWMA and Low-pass Filtering")
plt.show()


In [ ]:
with open("dataset/day_sample/20240825060442n.pkl", 'rb') as f:  # Open file in binary read mode
    pk = pickle.load(f)
    signal_sample = pk["signal"]  # Deserialize the signal data
    label_sample = pk["label"]
    print(label_sample)

signal_plot = signal_sample[0,:,1]
# Plot
plt.figure(figsize=(20, 5))
x = np.arange(signal_plot.shape[0])
plt.plot(x, signal_plot)
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.title("Signal")
plt.show()




## Labels

In [ ]:
from datetime import datetime, timedelta
import csv

In [ ]:
# Labels
fps = 25
SITE = "site"  # site identifier used in the CV output file names (anonymised)

# Folder path
folder_path = Path("data/labels_json")  

# Unique file names
file_names = {file.name[:19] for file in folder_path.iterdir() if file.is_file()}
label_dict = {}

# Output dataframe
data_rows = []
# df = pd.DataFrame(columns=["start", "end", "type", "x"])

for prefix in file_names:
    # Load the data
    with open(folder_path / f"{prefix}_{SITE}_xs_cat_vehicles.json", "r") as f:
        j = json.load(f)
        vehicles = j["data"]
        header = j["header"]

    # Load the video time
    video_time = datetime.strptime(header["video_time"], "%Y-%m-%d %H:%M:%S")
    if video_time.hour < 8 or video_time.hour > 17:
        continue

    with open(folder_path / f"{prefix}_{SITE}_xs_cat_trajectories.json", "r") as f:
        traj = json.load(f)["data"]

    for k,v in vehicles.items():
        if v["axle"] <= 2: # Only more than 2 axles can be considered as a truck?
            v["type"] = 1
        first_frame = traj[k][0][0]/fps - traj[k][0][2] / (v["speed"]/3.6)
        time_start = video_time + timedelta(seconds=first_frame)
        time_end = time_start + timedelta(seconds=25/(v["speed"]/3.6) ) 
        x1 = traj[k][0][1]
        
        # Add a new row
        data_rows.append([time_start, time_end, v["type"], x1, v["speed"], v["axle"]])
        # df.loc[len(df)] = [time_start, time_end, v["type"], x1]
        # x2 = traj[k][-1][1]
        # print(f"Vehicle {k} starts at {time_start} and ends at {times_end} with speed {v["speed"]} km/h. The vehicle is in lane {x1} to {x2}.")

df_label = pd.DataFrame(data_rows, columns=["start", "end", "type", "x", "speed", "axle"])
df_label = df_label.sort_values(by="start")    
df_label.to_csv("data/label_axle_corr.csv", index=False)
# df_label.to_feather("data/label_axle.feather")

In [ ]:
df_label = pd.read_csv("data/label_axle_correct.csv", parse_dates=["start", "end"])
print(df_label.head())
df_label["time_delta"] = 35 / (df_label["speed"] / 3.6)
df_label["time_delta"] = pd.to_timedelta(df_label["time_delta"], unit="s")
df_label["start"] = df_label["start"] - pd.Timedelta(seconds=1.9) - df_label["time_delta"]
df_label["end"] = df_label["end"] - pd.Timedelta(seconds=1.9) - df_label["time_delta"]
df_label = df_label.drop(columns=["time_delta"])

print(df_label.head())
df_label.to_csv("data/label_axle_corr.csv", index=False)

## Label analysis

In [ ]:
df_label = pd.read_csv("data/label_axle_corr.csv", parse_dates=["start", "end"])
df_label["x_greater_than_6"] = df_label["x"] > 7

result = pd.pivot_table(df_label, 
                        index="type", 
                        columns="x_greater_than_6", 
                        aggfunc="size", 
                        fill_value=0)

result.columns = ["x <= 7", "x > 7"]
print(result)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# Load the data
df_plot = pd.read_csv("data/label_axle_corr.csv")

# Calculate KDE (Kernel Density Estimation)
x_vals = np.linspace(df_plot["x"].min(), df_plot["x"].max(), 1000)  # Generate evenly spaced values for x
kde = gaussian_kde(df_plot["x"])  # Compute KDE for the x column
kde_vals = kde(x_vals)  # Evaluate KDE values for the generated x values

# Find the minimum point of the KDE curve in the central region
center_mask = (x_vals > np.percentile(df_plot["x"], 25)) & (x_vals < np.percentile(df_plot["x"], 75))  # Define the central region
center_x = x_vals[center_mask]  # Extract x values in the central region
center_kde = kde_vals[center_mask]  # Extract KDE values in the central region
min_idx = np.argmin(center_kde)  # Find the index of the minimum KDE value in the central region
min_x = center_x[min_idx]  # Get the x-coordinate of the minimum point
min_y = center_kde[min_idx]  # Get the y-coordinate of the minimum point

# Plot the KDE curve
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df_plot, x="x", fill=True, alpha=0.6, label="KDE")  # Plot the KDE curve
plt.scatter(min_x, min_y, color='red', zorder=3, label="Minimum in Center")  # Highlight the minimum point
plt.axvline(min_x, linestyle="--", color="gray", alpha=0.6)  # Add a vertical line at the minimum point
plt.title("KDE of x with Central Minimum")  # Add a title
plt.legend()  # Add a legend
plt.show()  # Display the plot

# Print the coordinates of the minimum point
print(f"Minimum point coordinates: x = {min_x:.4f}, y = {min_y:.4f}")


## Peak Detection Label

In [ ]:
# Peak detection
from glob import glob
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import numpy as np

count_list = glob("data/raw/counts/load_counts/*.csv")  # counts of the on-site peak detector (not public)
count_df_list = []
for count_path in count_list:
    start_day = count_path.split("_")[-3][:8]
    if start_day == "20240830" or start_day == "20240831":
        df = pd.read_csv(count_path, parse_dates=["Timestamp"])
        count_df_list.append(df[['Timestamp', 'type', 'lane']])
    else:
        continue
    
merged_df = pd.concat(count_df_list, ignore_index=True)
print(merged_df.head())
print(merged_df.shape)

# df_label = pd.read_csv("data/label_axle_corr.csv", parse_dates=["start", "end"])

In [ ]:
file_list = glob("dataset/test_folder_daytime/*.pkl")  # Get a list of all .pkl files in the specified folder
out_folder = Path("runs/peak_count")  # Define the output folder path
out_folder.mkdir(parents=True, exist_ok=True)  # Create the output folder if it doesn't exist

# Iterate through each file in the file list
for file_name in file_list:

    count_arr = np.zeros((720, 4))  # Initialize a 2D array to store counts for 719 time intervals and 4 categories
    fname = Path(file_name).stem  # Extract the file name without extension
    # Parse the file name to extract the timestamp
    timestamp_str = fname[:14]  # Assume the first 14 characters of the file name represent the timestamp
    file_timestamp = datetime.strptime(timestamp_str, "%Y%m%d%H%M%S")  # Convert the timestamp string to a datetime object
    print(file_timestamp)
    
    # Generate 5-second intervals starting from the file timestamp
    periods = [file_timestamp + timedelta(seconds=5 * i) for i in range(719)]  # Create 719 intervals of 5 seconds each

    # Count the rows in merged_df that fall within each time interval
    for idx, period_start in enumerate(periods):
        period_end = period_start + timedelta(seconds=5)  # Define the end of the current interval
        
        # Filter rows in merged_df that fall within the current time interval
        mask = (merged_df['Timestamp'] >= period_start) & (merged_df['Timestamp'] < period_end)
        for lane, group in merged_df[mask].groupby("lane"):  # Group the filtered rows by lane
            if lane == "lane2":  # If the lane is "lane2"
                count_arr[idx, 0] += len(group[group["type"] == "car"])  # Count cars in lane2
                count_arr[idx, 2] += len(group[group["type"] == "truck"])  # Count trucks in lane2
            elif lane == "lane1":  # If the lane is "lane1"
                count_arr[idx, 1] += len(group[group["type"] == "car"])  # Count cars in lane1
                count_arr[idx, 3] += len(group[group["type"] == "truck"])  # Count trucks in lane1
        
    # Save the count array as a .npy file in the output folder
    np.save(out_folder / f"{fname}_norm.npy", count_arr)
